# Rust — Weather Data Analysis

This notebook covers:
1. Loading `weather.json` with `serde_json`
2. Data inspection
3. Data cleaning (type coercion, missing values)
4. Descriptive statistics
5. Outlier detection (IQR & Z-score)
6. Data processing & filtering
7. Correlation analysis

> **Runtime:** This notebook requires the [evcxr](https://github.com/evcxr/evcxr) Rust kernel.  
> See the `.devcontainer/` folder to run this in GitHub Codespaces with zero local setup.

## Setup — Load Dependencies

The `:dep` directive tells evcxr to download and compile crates from crates.io.  
Run this cell **first** — it may take ~30 seconds on first run while Cargo compiles.

In [ ]:
// evcxr downloads and compiles crates with :dep (serde_json already pulls in serde)
:dep serde_json = "1"

use serde_json::Value;
use std::fs;

println!("Dependencies loaded.");

## 1. Loading Weather Data

We read `weather.json` with `std::fs` and parse it using `serde_json`.  
Each record is kept as a `serde_json::Value` so we can handle the mixed types present in the raw data.

In [ ]:
// weather.json must be in the directory the kernel is launched from
let raw = fs::read_to_string("weather.json")
    .expect("Could not read weather.json -- make sure it is in the same directory");

let records: Vec<Value> = serde_json::from_str(&raw).expect("JSON parse error");

println!("Records loaded : {}", records.len());

// Column names taken from the first record.
// They are cloned into owned Strings because evcxr requires 'static types (no borrows from `records`).
let columns: Vec<String> = records[0]
    .as_object().unwrap()
    .keys()
    .cloned()
    .collect();
println!("Columns ({}): {:?}", columns.len(), columns);

## 2. Data Inspection

Before cleaning, we print the first few records and identify which columns contain numeric data stored as strings.

In [ ]:
// Print the first 3 records as pretty-printed JSON
println!("--- First 3 records ---");
for r in records.iter().take(3) {
    println!("{}", serde_json::to_string_pretty(r).unwrap());
}

In [ ]:
// Count, per column, how many values are JSON numbers and how many are strings.
// Columns with #String > 0 need conversion before they can be analysed.
let check_cols = [
    "MinTemp","MaxTemp","Rainfall","Evaporation","Sunshine",
    "WindGustSpeed","WindSpeed9am","WindSpeed3pm",
    "Humidity9am","Humidity3pm","Pressure9am","Pressure3pm",
    "Temp9am","Temp3pm","RISK_MM"
];

println!("{:<16} {:>10} {:>10}", "Column", "#Numeric", "#String");
println!("{}", "-".repeat(38));
for col in &check_cols {
    let (mut nums, mut strs) = (0usize, 0usize);
    for r in &records {
        match &r[col] {
            Value::Number(_) => nums += 1,
            Value::String(_) => strs += 1,
            _ => {}   // null or missing field: not counted
        }
    }
    println!("{:<16} {:>10} {:>10}", col, nums, strs);
}

## 3. Data Cleaning

### 3.1 Type Coercion
Several columns (`Sunshine`, `WindGustSpeed`, `WindSpeed9am`) store numbers as JSON strings.  
We define a helper `to_f64()` that handles both cases transparently.

### 3.2 Missing Values
We count `null` and unparseable values per column and decide how to handle them:  
- **Drop** rows with missing values in key modelling columns  
- **Impute** with the column mean for less critical columns

In [ ]:
// Convert a Value to f64 whether it is a number or a string ("6.3").
// Returns None if the value is null, missing or cannot be parsed.
fn to_f64(v: &Value) -> Option<f64> {
    match v {
        Value::Number(n) => n.as_f64(),
        Value::String(s) => s.trim().parse::<f64>().ok(),
        _ => None,
    }
}

// Extract a full column, keeping missing values as None
fn extract_col(records: &[Value], col: &str) -> Vec<Option<f64>> {
    records.iter().map(|r| to_f64(&r[col])).collect()
}

// Extract a column dropping missing values (flatten removes the Nones)
fn clean_col(records: &[Value], col: &str) -> Vec<f64> {
    extract_col(records, col).into_iter().flatten().collect()
}

println!("Helpers defined.");

In [ ]:
// Missing-value report after type coercion
println!("{:<16} {:>8} {:>8} {:>10}", "Column", "Total", "Missing", "Missing%");
println!("{}", "-".repeat(44));
for col in &check_cols {
    let vals = extract_col(&records, col);
    let missing = vals.iter().filter(|v| v.is_none()).count();
    let pct = missing as f64 / vals.len() as f64 * 100.0;
    println!("{:<16} {:>8} {:>8} {:>9.1}%", col, vals.len(), missing, pct);
}

In [ ]:
// Mean imputation: replace each None with the average of the present values
fn impute_mean(vals: &[Option<f64>]) -> Vec<f64> {
    let clean: Vec<f64> = vals.iter().flatten().copied().collect();
    let mean = clean.iter().sum::<f64>() / clean.len() as f64;
    vals.iter().map(|v| v.unwrap_or(mean)).collect()
}

// Example with Evaporation
let evap_raw = extract_col(&records, "Evaporation");
let evap = impute_mean(&evap_raw);
println!("Evaporation — raw missing: {}", evap_raw.iter().filter(|v| v.is_none()).count());
println!("Evaporation — after imputation, len: {}, first 5: {:?}", evap.len(), &evap[..5]);

## 4. Descriptive Statistics

For each numeric column we compute: **n**, **mean**, **median**, **std dev**, **min**, **max**, **skewness**, and **kurtosis**.

In [ ]:
fn mean(v: &[f64]) -> f64 { v.iter().sum::<f64>() / v.len() as f64 }

// Median: middle value (or the average of the two middle values if n is even)
fn median(v: &[f64]) -> f64 {
    let mut s = v.to_vec();
    s.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let n = s.len();
    if n % 2 == 0 { (s[n/2-1] + s[n/2]) / 2.0 } else { s[n/2] }
}

// Population standard deviation (divides by n, not n - 1)
fn std_dev(v: &[f64]) -> f64 {
    let m = mean(v);
    (v.iter().map(|x| (x - m).powi(2)).sum::<f64>() / v.len() as f64).sqrt()
}

// Skewness: average of the cubed z-scores (0 = symmetric)
fn skewness(v: &[f64]) -> f64 {
    let m = mean(v);
    let s = std_dev(v);
    v.iter().map(|x| ((x - m) / s).powi(3)).sum::<f64>() / v.len() as f64
}

// Excess kurtosis: average of the z-scores to the fourth power minus 3 (normal = 0)
fn kurtosis(v: &[f64]) -> f64 {
    let m = mean(v);
    let s = std_dev(v);
    v.iter().map(|x| ((x - m) / s).powi(4)).sum::<f64>() / v.len() as f64 - 3.0
}

println!("Statistics functions defined.");

In [ ]:
let stat_cols = [
    "MinTemp","MaxTemp","Rainfall","Humidity9am",
    "Humidity3pm","Pressure9am","Temp9am","Temp3pm"
];

println!("{:<14} {:>6} {:>7} {:>7} {:>6} {:>7} {:>7} {:>7} {:>7}",
         "Column","n","mean","median","std","min","max","skew","kurt");
println!("{}", "-".repeat(72));
for col in &stat_cols {
    let v = clean_col(&records, col);
    // Sort a copy just to read the minimum (first) and maximum (last)
    let mut sorted = v.clone();
    sorted.sort_by(|a, b| a.partial_cmp(b).unwrap());
    println!("{:<14} {:>6} {:>7.2} {:>7.2} {:>6.2} {:>7.2} {:>7.2} {:>7.3} {:>7.3}",
        col, v.len(), mean(&v), median(&v), std_dev(&v),
        sorted[0], sorted[v.len()-1], skewness(&v), kurtosis(&v));
}

## 5. Outlier Detection

### 5.1 IQR Method
A value is an outlier if it falls below $Q_1 - 1.5 \times IQR$ or above $Q_3 + 1.5 \times IQR$.

### 5.2 Z-score Method
A value is an outlier if $|z| > threshold$ (commonly 3.0).

In [ ]:
// IQR bounds: [Q1 - 1.5*IQR, Q3 + 1.5*IQR].
// Quartiles are approximated by position in the sorted vector (no interpolation).
fn iqr_bounds(v: &[f64]) -> (f64, f64) {
    let mut s = v.to_vec();
    s.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let n = s.len();
    let q1 = s[n / 4];
    let q3 = s[3 * n / 4];
    let iqr = q3 - q1;
    (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
}

// Values that fall outside the bounds
fn iqr_outliers(v: &[f64]) -> Vec<f64> {
    let (lo, hi) = iqr_bounds(v);
    v.iter().copied().filter(|&x| x < lo || x > hi).collect()
}

println!("{:<14} {:>10} {:>10} {:>10} {:>10}",
         "Column", "Lower", "Upper", "Outliers", "Outl%");
println!("{}", "-".repeat(56));
for col in &stat_cols {
    let v = clean_col(&records, col);
    let (lo, hi) = iqr_bounds(&v);
    let out = iqr_outliers(&v);
    let pct = out.len() as f64 / v.len() as f64 * 100.0;
    println!("{:<14} {:>10.2} {:>10.2} {:>10} {:>9.1}%",
             col, lo, hi, out.len(), pct);
}

In [ ]:
// Returns (index, value, z-score) for every observation with |z| > threshold
fn zscore_outliers(v: &[f64], threshold: f64) -> Vec<(usize, f64, f64)> {
    let m = mean(v);
    let s = std_dev(v);
    v.iter().enumerate()
        .filter_map(|(i, &x)| {
            let z = (x - m) / s;
            if z.abs() > threshold { Some((i, x, z)) } else { None }
        })
        .collect()
}

// Rainfall is usually heavily skewed, so it produces many outliers
let rainfall = clean_col(&records, "Rainfall");
let outliers = zscore_outliers(&rainfall, 3.0);
println!("Rainfall — Z-score outliers (|z| > 3.0): {}", outliers.len());
println!("{:<8} {:>10} {:>10}", "Index", "Value", "Z-score");
println!("{}", "-".repeat(30));
for (i, val, z) in &outliers {
    println!("{:<8} {:>10.2} {:>10.3}", i, val, z);
}

In [ ]:
// Remove the IQR outliers from Rainfall and compare statistics before and after
let rainfall = clean_col(&records, "Rainfall");
let (lo, hi) = iqr_bounds(&rainfall);
let rainfall_clean: Vec<f64> = rainfall.iter().copied().filter(|&x| x >= lo && x <= hi).collect();

println!("Rainfall — before outlier removal:");
println!("  n={}, mean={:.2}, std={:.2}, min={:.2}, max={:.2}",
    rainfall.len(), mean(&rainfall), std_dev(&rainfall),
    rainfall.iter().cloned().fold(f64::INFINITY, f64::min),
    rainfall.iter().cloned().fold(f64::NEG_INFINITY, f64::max));

println!("\nRainfall — after IQR outlier removal:");
println!("  n={}, mean={:.2}, std={:.2}, min={:.2}, max={:.2}",
    rainfall_clean.len(), mean(&rainfall_clean), std_dev(&rainfall_clean),
    rainfall_clean.iter().cloned().fold(f64::INFINITY, f64::min),
    rainfall_clean.iter().cloned().fold(f64::NEG_INFINITY, f64::max));

println!("\nRows removed: {}", rainfall.len() - rainfall_clean.len());

## 6. Data Processing & Filtering

### 6.1 Min-Max Normalization
Rescales values to $[0, 1]$: $x' = \frac{x - x_{min}}{x_{max} - x_{min}}$

### 6.2 Filtering
Select subsets of records by condition (e.g. rainy days, extreme heat).

In [ ]:
// Min-max normalisation: rescales values to the range [0, 1]
fn min_max_normalize(v: &[f64]) -> Vec<f64> {
    let min = v.iter().cloned().fold(f64::INFINITY, f64::min);
    let max = v.iter().cloned().fold(f64::NEG_INFINITY, f64::max);
    v.iter().map(|&x| (x - min) / (max - min)).collect()
}

// Standardisation (z-score): mean 0 and standard deviation 1
fn z_standardize(v: &[f64]) -> Vec<f64> {
    let m = mean(v);
    let s = std_dev(v);
    v.iter().map(|&x| (x - m) / s).collect()
}

let max_temp = clean_col(&records, "MaxTemp");
let norm = min_max_normalize(&max_temp);
let std_vals = z_standardize(&max_temp);

println!("MaxTemp — original first 5: {:?}", &max_temp[..5]);
println!("MaxTemp — normalized  first 5: {:.3?}", &norm[..5]);
println!("MaxTemp — standardized first 5: {:.3?}", &std_vals[..5]);

In [ ]:
// Only the records that meet each condition are counted.
// No Vec<&Value> is stored because it would borrow from `records`, which evcxr rejects ('static).
let n_rainy = records.iter()
    .filter(|r| r["RainToday"].as_str() == Some("Yes"))
    .count();

let n_hot = records.iter()
    .filter(|r| to_f64(&r["MaxTemp"]).map_or(false, |t| t > 35.0))
    .count();

let n_humid = records.iter()
    .filter(|r| to_f64(&r["Humidity3pm"]).map_or(false, |h| h > 80.0))
    .count();

let n = records.len() as f64;
println!("Total records     : {}", records.len());
println!("Rainy days        : {} ({:.1}%)", n_rainy, n_rainy as f64 / n * 100.0);
println!("Hot days (>35 C)  : {} ({:.1}%)", n_hot,   n_hot   as f64 / n * 100.0);
println!("High humidity >80%: {} ({:.1}%)", n_humid,  n_humid  as f64 / n * 100.0);

In [ ]:
// Compare MaxTemp between rainy and dry days
let rainy_max: Vec<f64> = records.iter()
    .filter(|r| r["RainToday"].as_str() == Some("Yes"))
    .filter_map(|r| to_f64(&r["MaxTemp"]))
    .collect();

let dry_max: Vec<f64> = records.iter()
    .filter(|r| r["RainToday"].as_str() == Some("No"))
    .filter_map(|r| to_f64(&r["MaxTemp"]))
    .collect();

println!("{:<12} {:>6} {:>8} {:>8} {:>8}",
         "RainToday", "n", "mean", "median", "std");
println!("{}", "-".repeat(44));
println!("{:<12} {:>6} {:>8.2} {:>8.2} {:>8.2}",
    "Yes", rainy_max.len(), mean(&rainy_max), median(&rainy_max), std_dev(&rainy_max));
println!("{:<12} {:>6} {:>8.2} {:>8.2} {:>8.2}",
    "No", dry_max.len(), mean(&dry_max), median(&dry_max), std_dev(&dry_max));

## 7. Correlation Analysis

We compute the **Pearson correlation coefficient** between pairs of numeric columns:

$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \cdot \sum (y_i - \bar{y})^2}}$$

In [ ]:
// Pearson correlation coefficient: normalised covariance, in the range [-1, 1]
fn pearson(xs: &[f64], ys: &[f64]) -> f64 {
    assert_eq!(xs.len(), ys.len(), "Vectors must have the same length");
    let mx = mean(xs);
    let my = mean(ys);
    let num: f64 = xs.iter().zip(ys.iter()).map(|(&x, &y)| (x - mx) * (y - my)).sum();
    let dx: f64 = xs.iter().map(|&x| (x - mx).powi(2)).sum::<f64>().sqrt();
    let dy: f64 = ys.iter().map(|&y| (y - my).powi(2)).sum::<f64>().sqrt();
    num / (dx * dy)
}

println!("Pearson correlation function defined.");

In [ ]:
// Align two columns row by row: only records where both have a value are kept
// (the `?` operator returns None from the closure if either is missing)
fn aligned_cols(records: &[Value], col_a: &str, col_b: &str) -> (Vec<f64>, Vec<f64>) {
    records.iter()
        .filter_map(|r| {
            let a = to_f64(&r[col_a])?;
            let b = to_f64(&r[col_b])?;
            Some((a, b))
        })
        .unzip()
}

let corr_cols = ["MinTemp","MaxTemp","Humidity3pm","Rainfall","Pressure9am","Temp3pm"];

// Header (names are truncated to 9 characters to fit the column)
print!("{:<12}", "");
for c in &corr_cols { print!("{:>10}", &c[..c.len().min(9)]); }
println!();
println!("{}", "-".repeat(12 + 10 * corr_cols.len()));

// Correlation matrix; the diagonal is 1 by definition
for row in &corr_cols {
    print!("{:<12}", &row[..row.len().min(11)]);
    for col in &corr_cols {
        if row == col {
            print!("{:>10.3}", 1.0_f64);
        } else {
            let (a, b) = aligned_cols(&records, row, col);
            print!("{:>10.3}", pearson(&a, &b));
        }
    }
    println!();
}

In [ ]:
// Most correlated pairs (|r| > 0.5), walking only the upper triangle
// of the matrix to avoid duplicate pairs and self-correlation
let all_cols = ["MinTemp","MaxTemp","Rainfall","Evaporation",
                "Humidity9am","Humidity3pm","Pressure9am","Pressure3pm",
                "Temp9am","Temp3pm"];

let mut pairs: Vec<(&str, &str, f64)> = Vec::new();
for i in 0..all_cols.len() {
    for j in (i+1)..all_cols.len() {
        let (a, b) = aligned_cols(&records, all_cols[i], all_cols[j]);
        let r = pearson(&a, &b);
        if r.abs() > 0.5 {
            pairs.push((all_cols[i], all_cols[j], r));
        }
    }
}
// Sort descending by |r|
pairs.sort_by(|a, b| b.2.abs().partial_cmp(&a.2.abs()).unwrap());

println!("Top correlated pairs (|r| > 0.5):");
println!("{:<14} {:<14} {:>8}", "Col A", "Col B", "r");
println!("{}", "-".repeat(38));
for (a, b, r) in &pairs {
    println!("{:<14} {:<14} {:>8.3}", a, b, r);
}